# 03 · Juntando as tabelas

Cada fonte possui suas próprias colunas. Usamos o **estado e o ano** para juntar informações de homicídios, leis, desemprego e renda. Depois criamos a coluna que o modelo vai tentar prever: **a taxa de homicídios do ano seguinte**. Conferimos se o processo perdeu algum estado ou criou registros repetidos.

In [ ]:
from pathlib import Path
import pandas as pd

base = Path("../data")

cdc = pd.read_csv(
    base / "raw/cdc_homicides_2019_2024.csv"
)

ccw = pd.read_csv(
    base / "processed/rand_ccw_2019_2024.csv"
)

background = pd.read_csv(
    base / "processed/rand_background_2019_2024.csv"
)

print("CDC:", cdc.shape)
print("CCW:", ccw.shape)
print("Background:", background.shape)

In [ ]:
cdc = cdc[cdc["name"] != "District of Columbia"].copy()

cdc = cdc.rename(columns={
    "name": "state",
    "period": "year",
    "count_sup": "homicide_count",
    "rate": "homicide_rate"
})

cdc["year"] = cdc["year"].astype(int)

print("CDC após padronização:", cdc.shape)

display(
    cdc[
        ["state", "year", "homicide_count", "homicide_rate"]
    ].head()
)

In [ ]:
chaves = ["state", "year"]

bases = {
    "CDC": cdc,
    "CCW": ccw,
    "Background": background
}

for nome, tabela in bases.items():
    print(f"\n{nome}")
    print("Registros:", len(tabela))
    print("Estados:", tabela["state"].nunique())
    print("Duplicações:", tabela.duplicated(chaves).sum())
    print("Chaves ausentes:", tabela[chaves].isna().sum().sum())

chaves_cdc = set(map(tuple, cdc[chaves].to_numpy()))
chaves_ccw = set(map(tuple, ccw[chaves].to_numpy()))
chaves_background = set(map(tuple, background[chaves].to_numpy()))

print("\nChaves CDC ausentes no CCW:", len(chaves_cdc - chaves_ccw))
print("Chaves CDC ausentes no Background:", len(chaves_cdc - chaves_background))
print("Chaves extras no CCW:", len(chaves_ccw - chaves_cdc))
print("Chaves extras no Background:", len(chaves_background - chaves_cdc))

In [ ]:
df_integrado = (
    cdc.merge(
        ccw,
        on=["state", "year"],
        how="inner",
        validate="one_to_one"
    )
    .merge(
        background,
        on=["state", "year"],
        how="inner",
        validate="one_to_one"
    )
)

print("Dimensões:", df_integrado.shape)

display(df_integrado.head())

In [ ]:
print("Registros:", len(df_integrado))

print("Estados:", df_integrado["state"].nunique())

print("Duplicações:",
      df_integrado.duplicated(["state", "year"]).sum())

print("\nValores ausentes:")
display(df_integrado.isna().sum())

In [ ]:
colunas = [
    "state",
    "year",
    "homicide_count",
    "homicide_rate",
    "ccw_shall_issue",
    "ccw_permitless",
    "ccw_may_issue",
    "ccw_prohibited",
    "background_handgun",
    "background_longgun"
]

df_modelo = df_integrado[colunas].copy()

df_modelo = df_modelo.sort_values(
    ["state", "year"]
).reset_index(drop=True)

print("Dimensões:", df_modelo.shape)
display(df_modelo.head(10))

In [ ]:
df_modelo["homicide_rate_next_year"] = (
    df_modelo
    .groupby("state")["homicide_rate"]
    .shift(-1)
)

display(
    df_modelo[
        ["state", "year", "homicide_rate",
         "homicide_rate_next_year"]
    ].head(12)
)

print(
    "Alvos ausentes:",
    df_modelo["homicide_rate_next_year"].isna().sum()
)

In [ ]:
df_modelo["next_year"] = (
    df_modelo.groupby("state")["year"].shift(-1)
)

intervalos_invalidos = df_modelo[
    df_modelo["next_year"].notna()
    & (df_modelo["next_year"] != df_modelo["year"] + 1)
]

print("Intervalos inválidos:", len(intervalos_invalidos))

display(intervalos_invalidos[
    ["state", "year", "next_year"]
].head(10))

In [ ]:
# Manter apenas registros com variável-alvo conhecida
df_supervisionado = df_modelo.dropna(
    subset=["homicide_rate_next_year"]
).copy()

# Remover a coluna auxiliar de validação
df_supervisionado = df_supervisionado.drop(
    columns=["next_year"]
)

print("Dimensões:", df_supervisionado.shape)

display(df_supervisionado.head())

In [ ]:
from pathlib import Path

destino = Path(
    "../data/processed/us_gun_policy_supervised_v1.csv"
)

df_supervisionado.to_csv(destino, index=False)

print("Salvo em:", destino.resolve())

In [ ]:
from pathlib import Path
import pandas as pd

arquivo = Path("../data/raw/staadata/staadata.xlsx")

excel = pd.ExcelFile(arquivo)

print("Abas disponíveis:")
print(excel.sheet_names)

for aba in excel.sheet_names:
    amostra = pd.read_excel(
        arquivo,
        sheet_name=aba,
        nrows=8,
        header=None
    )

    print(f"\nAba: {aba}")
    display(amostra)

In [ ]:
bls = pd.read_excel(
    arquivo,
    sheet_name=excel.sheet_names[0],
    header=None,
    skiprows=8,
    usecols=[0, 1, 2, 9],
    names=["fips", "state", "year", "unemployment_rate"]
)

print("Dimensões:", bls.shape)

display(bls.head(10))

print("\nTipos de dados:")
print(bls.dtypes)

In [ ]:
# Utilizar os estados da base CDC como referência
cdc = pd.read_csv(
    "../data/raw/cdc_homicides_2019_2024.csv"
)

estados_validos = set(cdc["name"]) - {"District of Columbia"}

bls["year"] = pd.to_numeric(
    bls["year"],
    errors="coerce"
)

bls_filtrado = bls[
    bls["state"].isin(estados_validos)
    & bls["year"].between(2019, 2024)
].copy()

print("Dimensões:", bls_filtrado.shape)
print("Estados:", bls_filtrado["state"].nunique())

display(bls_filtrado.groupby("year")["state"].nunique())

In [ ]:
# Validação final da base BLS
print("Duplicações:",
      bls_filtrado.duplicated(["state", "year"]).sum())

print("Taxas ausentes:",
      bls_filtrado["unemployment_rate"].isna().sum())

print("Taxas fora do intervalo:",
      (~bls_filtrado["unemployment_rate"].between(0, 100)).sum())

print("\nEstatísticas descritivas:")
display(bls_filtrado["unemployment_rate"].describe())

In [ ]:
from pathlib import Path

bls_final = bls_filtrado[
    ["state", "year", "unemployment_rate"]
].copy()

bls_final["year"] = bls_final["year"].astype(int)

destino = Path("../data/processed/bls_unemployment_2019_2024.csv")

bls_final.to_csv(destino, index=False)

print("Dimensões:", bls_final.shape)
print("Arquivo salvo:", destino.resolve())

display(bls_final.head())

In [ ]:
bls = pd.read_csv(
    "../data/processed/bls_unemployment_2019_2024.csv"
)

df_v2 = df_supervisionado.merge(
    bls,
    on=["state", "year"],
    how="left",
    validate="one_to_one"
)

print("Dimensões:", df_v2.shape)
print("Desemprego ausente:", df_v2["unemployment_rate"].isna().sum())

display(
    df_v2[
        ["state", "year", "homicide_rate",
         "unemployment_rate", "homicide_rate_next_year"]
    ].head(10)
)

In [ ]:
from pathlib import Path

destino = Path("../data/processed/us_gun_policy_supervised_v2.csv")

df_v2.to_csv(destino, index=False)

print("Arquivo salvo:", destino.resolve())

In [ ]:
bea_income = pd.read_csv(
    "../data/processed/bea_income_2019_2024.csv"
)

df_v3 = df_v2.merge(
    bea_income,
    on=["state", "year"],
    how="left",
    validate="one_to_one"
)

print("Dimensões:", df_v3.shape)
print("Rendas ausentes:", df_v3["income_per_capita"].isna().sum())

display(
    df_v3[
        [
            "state",
            "year",
            "homicide_rate",
            "unemployment_rate",
            "income_per_capita",
            "homicide_rate_next_year"
        ]
    ].head(10)
)

In [ ]:
print("Dimensões:", df_v3.shape)

print("Duplicações:",
      df_v3.duplicated(["state", "year"]).sum())

print("Valores ausentes por coluna:")
display(df_v3.isna().sum())

print("Estados:", df_v3["state"].nunique())
print("Anos:", sorted(df_v3["year"].unique()))
assert df_v3.shape == (250,13)
assert not df_v3.duplicated(["state","year"]).any()
assert not df_v3.isna().any().any()


In [ ]:
from pathlib import Path

destino = Path("../data/processed/us_gun_policy_supervised_v3.csv")

df_v3.to_csv(destino, index=False)

print("Dataset V3 salvo:", destino.resolve())
print("Dimensões:", df_v3.shape)